# Portfolio Optimization Project: V1 Implementation

This notebook implements Version 1 (V1) of a binary portfolio optimization system, focusing on an objective function and a cardinality constraint. It is designed to be beginner-friendly and mathematically verifiable.

## Section 1: Imports

In [23]:
# Import necessary libraries
import numpy as np
import pandas as pd
import itertools
import matplotlib.pyplot as plt # Optional, for basic plotting if desired

## Section 2: Load uploaded files

In [24]:
import pandas as pd
from google.colab import files
import io

print("Please upload 'mu.csv'.")
# Upload mu.csv
uploaded_mu = files.upload()
# Get the actual filename Colab used for mu.csv (e.g., 'mu.csv' or 'mu (1).csv')
mu_actual_filename = list(uploaded_mu.keys())[0]
mu_file_content = uploaded_mu[mu_actual_filename].decode('utf-8')
mu = pd.read_csv(io.StringIO(mu_file_content), index_col=0).squeeze() # Squeeze to get a Series/1D array

print("Please upload 'Sigma.csv'.")
# Upload Sigma.csv
uploaded_sigma = files.upload()
# Get the actual filename Colab used for Sigma.csv
sigma_actual_filename = list(uploaded_sigma.keys())[0]
sigma_file_content = uploaded_sigma[sigma_actual_filename].decode('utf-8')
Sigma = pd.read_csv(io.StringIO(sigma_file_content), index_col=0)

print("Files uploaded and loaded successfully.")

Please upload 'mu.csv'.


Saving mu.csv to mu (4).csv
Please upload 'Sigma.csv'.


Saving Sigma.csv to Sigma (5).csv
Files uploaded and loaded successfully.


In [25]:
# Perform sanity checks

# Print mu shape
print(f"mu shape: {mu.shape}")

# Print Sigma shape
print(f"Sigma shape: {Sigma.shape}")

# Check for missing values
if mu.isnull().any() or Sigma.isnull().any().any():
    print("Warning: Missing values detected in mu or Sigma.")
else:
    print("No missing values detected.")

# Expected shapes
expected_mu_shape = (20,)
expected_sigma_shape = (20, 20)

if mu.shape == expected_mu_shape and Sigma.shape == expected_sigma_shape:
    print(f"Shapes match expected: mu {expected_mu_shape}, Sigma {expected_sigma_shape}.")
else:
    print(f"Warning: Shapes do not match expected. Expected mu {expected_mu_shape}, Sigma {expected_sigma_shape}. Actual mu {mu.shape}, Sigma {Sigma.shape}.")

mu shape: (20,)
Sigma shape: (20, 20)
No missing values detected.
Shapes match expected: mu (20,), Sigma (20, 20).


## Section 3: Define frozen parameters

In [26]:
# Define frozen parameters for the optimization problem

# K: Target portfolio size (number of stocks to select)
K = 6
print(f"Target portfolio size (K): {K}")

# lambda_values: Risk-aversion parameters to sweep through
# Lower lambda means more emphasis on expected return, higher lambda emphasizes diversification.
lambda_values = [0.1, 0.5, 1.0, 2.0]
print(f"Lambda values for sweep: {lambda_values}")

# alpha: Cardinality penalty coefficient
# This parameter penalizes portfolios that do not meet the target cardinality K.
alpha = 20
print(f"Cardinality penalty coefficient (alpha): {alpha}")

# Number of stocks (derived from mu length, assuming it's consistent with Sigma)
n = len(mu)
print(f"Total number of available stocks (n): {n}")

Target portfolio size (K): 6
Lambda values for sweep: [0.1, 0.5, 1.0, 2.0]
Cardinality penalty coefficient (alpha): 20
Total number of available stocks (n): 20


## Section 4: Define energy/objective function

In [27]:
def energy(x, mu, Sigma, lam, alpha, K):
    """
    Computes the energy (objective function) of a given portfolio x.

    The objective function is defined as:
    E(x) = -\muᵀx + \lambda xᵀΣx + \alpha (\sum_i x_i - K)²

    Args:
        x (np.array): A binary vector representing the portfolio (1 if stock is selected, 0 otherwise).
        mu (np.array): Expected return vector.
        Sigma (np.array): Covariance matrix.
        lam (float): Risk-aversion parameter.
        alpha (float): Cardinality penalty coefficient.
        K (int): Target portfolio size.

    Returns:
        float: The computed energy of the portfolio.
    """
    mu = np.array(mu)
    Sigma = np.array(Sigma)
    x = np.array(x)
    x = np.array(x) # Ensure x is a numpy array for vector operations

    # Term 1: Negative expected return (we want to maximize return, so minimize negative return)
    # -μᵀx
    term1 = -np.dot(mu, x)

    # Term 2: Risk component (variance of the portfolio)
    # λ xᵀΣx
    term2 = lam * np.dot(x.T, np.dot(Sigma, x))

    # Term 3: Cardinality penalty
    # α (Σ_i x_i - K)²
    # This term penalizes portfolios whose selected number of stocks (sum(x)) deviates from K.
    current_cardinality = np.sum(x)
    term3 = alpha * (current_cardinality - K)**2

    # Total energy
    total_energy = term1 + term2 + term3

    return total_energy

<>:6: SyntaxWarning: invalid escape sequence '\m'
<>:6: SyntaxWarning: invalid escape sequence '\m'
/tmp/ipykernel_1560/3229010397.py:6: SyntaxWarning: invalid escape sequence '\m'
  E(x) = -\muᵀx + \lambda xᵀΣx + \alpha (\sum_i x_i - K)²


## Section 5: Tiny verification stage

In [28]:
# This tiny verification stage is crucial for ensuring the objective function and basic logic
# are correctly implemented before running on the full dataset. It allows us to manually
# check results for a small, exhaustive search space.

# Take only the first 5 stocks for this small test
num_stocks_small = 5
mu_small = mu[:num_stocks_small]
Sigma_small = Sigma.iloc[:num_stocks_small, :num_stocks_small]

# Set a smaller target portfolio size for the test
K_small = 2

# Use an arbitrary lambda for verification
lam_small = 0.5

print(f"--- Tiny Verification Stage (first {num_stocks_small} stocks, K={K_small}, lambda={lam_small}) ---")
print(f"mu_small shape: {mu_small.shape}")
print(f"Sigma_small shape: {Sigma_small.shape}")

# Generate all binary portfolios for these 5 stocks (2^5 = 32 possibilities)
# Using itertools.product for a brute-force check on a small scale.
all_portfolios_small = list(itertools.product([0, 1], repeat=num_stocks_small))

min_energy_small = float('inf')
best_portfolio_small = None

# Evaluate all 32 possibilities
for portfolio_x in all_portfolios_small:
    current_energy = energy(portfolio_x, mu_small, Sigma_small, lam_small, alpha, K_small)
    if current_energy < min_energy_small:
        min_energy_small = current_energy
        best_portfolio_small = portfolio_x

stocks = [
    "AAPL","MSFT","NVDA","GOOGL","AMZN",
    "JPM","V","MA","GS","BAC",
    "LLY","JNJ","MRK","PFE","ABBV",
    "XOM","CVX","GE","CAT","HON"
]

selected_indices_small = [i for i, x_val in enumerate(best_portfolio_small) if x_val == 1]
selected_stocks_names_small = [
    stocks[i] for i in selected_indices_small
] # Assuming generic stock names
cardinality_small = np.sum(best_portfolio_small)

print("\nVerification Results:")
print(f"Best portfolio (binary representation): {best_portfolio_small}")
print(f"Selected stocks (indices): {selected_indices_small}")
print(f"Selected stocks (names): {selected_stocks_names_small}")
print(f"Minimum energy: {min_energy_small:.4f}")
print(f"Cardinality: {cardinality_small}")
print(f"Expected cardinality (K_small): {K_small}")

if cardinality_small == K_small:
    print("Cardinality constraint is satisfied for the best portfolio in verification (good).")
else:
    print("Cardinality constraint is NOT satisfied for the best portfolio in verification (check alpha).")

--- Tiny Verification Stage (first 5 stocks, K=2, lambda=0.5) ---
mu_small shape: (5,)
Sigma_small shape: (5, 5)

Verification Results:
Best portfolio (binary representation): (1, 0, 0, 0, 1)
Selected stocks (indices): [0, 4]
Selected stocks (names): ['AAPL', 'AMZN']
Minimum energy: -0.0055
Cardinality: 2
Expected cardinality (K_small): 2
Cardinality constraint is satisfied for the best portfolio in verification (good).


## Section 6: Full V1 implementation

In [29]:
# For the full V1 implementation, we will NOT brute force all 2^n portfolios.
# Instead, we generate ONLY feasible portfolios satisfying `sum(x) == K`.
# This is done using `itertools.combinations`, which is computationally much smarter
# for sparse binary vectors with a fixed number of ones.

print(f"--- Full V1 Implementation (n={n} stocks, target K={K}) ---")

# Generate all combinations of K stock indices out of n available stocks.
# Each combination represents a feasible portfolio that satisfies sum(x) == K.

# This function will be called repeatedly in the lambda sweep, so we'll define a helper here.
def find_best_portfolio_for_lambda(current_lambda, mu, Sigma, alpha, K, n):
    """
    Finds the best portfolio for a given lambda by enumerating all K-stock combinations.
    """
    min_energy_full = float('inf')
    best_portfolio_indices = None

    # Iterate through all combinations of K stocks out of n total stocks.
    # Each combination represents the indices of stocks selected for the portfolio.
    for indices_tuple in itertools.combinations(range(n), K):
        # Create a binary portfolio vector 'x' from the selected indices.
        # All elements are 0 by default, then set 1 at selected indices.
        x = np.zeros(n, dtype=int)
        for idx in indices_tuple:
            x[idx] = 1

        # Evaluate the energy of the current portfolio
        current_energy = energy(x, mu, Sigma, current_lambda, alpha, K)

        # If this portfolio has lower energy, it's our new best.
        if current_energy < min_energy_full:
            min_energy_full = current_energy
            best_portfolio_indices = indices_tuple

    # Convert best_portfolio_indices to a readable list of stock names (e.g., 'Stock_0', 'Stock_1')
    selected_stocks = [
    stocks[i] for i in best_portfolio_indices
]
    return selected_stocks, min_energy_full

--- Full V1 Implementation (n=20 stocks, target K=6) ---


## Section 7: λ sweep

In [30]:
# Now, we run the full V1 implementation for each specified lambda value.
# The results will be stored in a pandas DataFrame for clear presentation.

results = [] # To store dictionaries of results for each lambda

print("--- Starting Lambda Sweep ---")

for current_lambda in lambda_values:
    print(f"\nOptimizing for lambda = {current_lambda}...")
    # Call the helper function to find the best portfolio for the current lambda
    selected_stocks, min_energy = find_best_portfolio_for_lambda(current_lambda, mu, Sigma, alpha, K, n)

    results.append({
        'lambda': current_lambda,
        'selected_stocks': selected_stocks,
        'energy': min_energy
    })
    print(f"  Best portfolio found: {selected_stocks}")
    print(f"  Minimum energy: {min_energy:.4f}")

# Create a pandas DataFrame from the collected results
results_df = pd.DataFrame(results)

print("\n--- Lambda Sweep Results ---")
# Print the results table clearly
print(results_df.to_string(index=False))

# Optional: Basic visualization of energy vs lambda (for insights)
# plt.figure(figsize=(8, 5))
# plt.plot(results_df['lambda'], results_df['energy'], marker='o')
# plt.title('Minimum Energy vs. Lambda')
# plt.xlabel('Lambda (Risk Aversion Parameter)')
# plt.ylabel('Minimum Energy')
# plt.grid(True)
# plt.show()

--- Starting Lambda Sweep ---

Optimizing for lambda = 0.1...
  Best portfolio found: ['AAPL', 'AMZN', 'MA', 'GS', 'ABBV', 'CVX']
  Minimum energy: -0.0153

Optimizing for lambda = 0.5...
  Best portfolio found: ['AAPL', 'AMZN', 'MA', 'GS', 'ABBV', 'HON']
  Minimum energy: -0.0139

Optimizing for lambda = 1.0...
  Best portfolio found: ['AMZN', 'MA', 'GS', 'LLY', 'ABBV', 'HON']
  Minimum energy: -0.0124

Optimizing for lambda = 2.0...
  Best portfolio found: ['AMZN', 'MA', 'GS', 'LLY', 'ABBV', 'HON']
  Minimum energy: -0.0097

--- Lambda Sweep Results ---
 lambda                 selected_stocks    energy
    0.1 [AAPL, AMZN, MA, GS, ABBV, CVX] -0.015277
    0.5 [AAPL, AMZN, MA, GS, ABBV, HON] -0.013873
    1.0  [AMZN, MA, GS, LLY, ABBV, HON] -0.012446
    2.0  [AMZN, MA, GS, LLY, ABBV, HON] -0.009727


## Section 8: Simple interpretation

In [31]:
print("--- Interpretation of Results ---")

# Access the first row (lowest lambda) and last row (highest lambda) for comparison
lowest_lambda_result = results_df.iloc[0]
highest_lambda_result = results_df.iloc[-1]

print(f"1. **Low Lambda (e.g., \u03BB = {lowest_lambda_result['lambda']}):**")
print(f"   At low \u03BB, the objective function places more weight on **maximizing expected return**.")
print(f"   The selected portfolio ({lowest_lambda_result['selected_stocks']}) will likely contain stocks with high individual expected returns, potentially accepting higher risk.")

print(f"\n2. **High Lambda (e.g., \u03BB = {highest_lambda_result['lambda']}):**")
print(f"   At high \u03BB, the objective function places more emphasis on **minimizing portfolio risk (covariance)**.")
print(f"   The selected portfolio ({highest_lambda_result['selected_stocks']}) will aim for lower overall variance, likely selecting stocks that are less correlated or individually less volatile.")

print("\nIn general, increasing \u03BB shifts the trade-off from return maximization towards risk minimization within the fixed cardinality constraint (K). This demonstrates the risk-aversion parameter's role in shaping the optimal portfolio composition.")

--- Interpretation of Results ---
1. **Low Lambda (e.g., λ = 0.1):**
   At low λ, the objective function places more weight on **maximizing expected return**.
   The selected portfolio (['AAPL', 'AMZN', 'MA', 'GS', 'ABBV', 'CVX']) will likely contain stocks with high individual expected returns, potentially accepting higher risk.

2. **High Lambda (e.g., λ = 2.0):**
   At high λ, the objective function places more emphasis on **minimizing portfolio risk (covariance)**.
   The selected portfolio (['AMZN', 'MA', 'GS', 'LLY', 'ABBV', 'HON']) will aim for lower overall variance, likely selecting stocks that are less correlated or individually less volatile.

In general, increasing λ shifts the trade-off from return maximization towards risk minimization within the fixed cardinality constraint (K). This demonstrates the risk-aversion parameter's role in shaping the optimal portfolio composition.


#Conformance Tests

In [32]:
print(mu.shape)
print(Sigma.shape)

(20,)
(20, 20)


In [34]:
print(mu.isna().sum())
print(Sigma.isna().sum().sum())

0
0


# TOY VERIFICATION (5 STOCKS)
#

In [58]:
small_n = 5
small_indices = [0, 5, 10, 15, 18]

In [59]:
mu_small = mu.iloc[small_indices]
Sigma_small = Sigma.iloc[
    small_indices,
    small_indices
]

stocks_small = [stocks[i] for i in small_indices]

Step 2: Freeze small problem

In [96]:
K_small = 2
lam_Small = 0

Step 3: Enumerate ALL possibilities

In [97]:
all_portfolios = list(
    itertools.product([0,1], repeat=small_n)
)

Step 4: Evaluate all portfolios

In [98]:
best_energy = float('inf')
best_portfolio = None

In [99]:
for portfolio in all_portfolios:

    portfolio = np.array(portfolio)

    e = energy(
        portfolio,
        mu_small,
        Sigma_small,
        lam_small,
        alpha,
        K_small
    )

    if e < best_energy:
        best_energy = e
        best_portfolio = portfolio

Step 5: Decode selected stocks

In [100]:
selected_indices = np.where(best_portfolio == 1)[0]

In [101]:
selected_stocks = [
    stocks_small[i]
    for i in selected_indices
]

Step 6: Print results

In [103]:
print("TOY VERIFICATION RESULTS")
print("="*40)

print("Selected stocks:")
print(selected_stocks)

print("\nCardinality:")
print(np.sum(best_portfolio))

print("\nEnergy:")
print(best_energy)

TOY VERIFICATION RESULTS
Selected stocks:
['AAPL', 'LLY']

Cardinality:
2

Energy:
-0.0033474261970486362


In [82]:
print("Mean Returns (mu):")
for stock, val in zip(stocks_small, mu_small):
    print(f"{stock}: {val:.6f}")

print("\nDiagonal of Covariance Matrix (risk proxy):")
for stock, val in zip(stocks_small, np.diag(Sigma_small)):
    print(f"{stock}: {val:.6f}")

Mean Returns (mu):
AAPL: 0.001866
JPM: 0.001389
LLY: 0.001648
XOM: 0.000050
CAT: -0.000325

Diagonal of Covariance Matrix (risk proxy):
AAPL: 0.000194
JPM: 0.000194
LLY: 0.000111
XOM: 0.000240
CAT: 0.000194
